In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpiler des circuits à distance avec le Qiskit Transpiler Service

> **Danger:** À partir du 18 juillet 2025, le service est en cours de migration pour prendre en charge la nouvelle plateforme IBM Quantum&reg; et n'est pas disponible. Pour les passes IA, tu peux utiliser le [mode local](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     Le service est une version bêta, susceptible de changer.
>     Si tu as des retours ou souhaites contacter l'équipe de développement, utilise ce canal [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Le Qiskit Transpiler Service fournit des capacités de transpilation dans le cloud. En plus des capacités de transpilation locales de Qiskit, tes tâches de transpilation peuvent bénéficier à la fois des ressources cloud d'IBM Quantum et des passes de transpilation alimentées par l'IA.

Le Qiskit Transpiler Service propose une bibliothèque Python pour intégrer de façon transparente ce service et ses fonctionnalités dans tes patterns et workflows Qiskit actuels. Ce service est uniquement disponible pour les utilisateurs des plans IBM Quantum Premium, Flex et On-Prem (via l'API IBM Quantum Platform).

<span id="install-transpiler-service"></span>
## Installer le package qiskit-ibm-transpiler
Pour utiliser le Qiskit Transpiler Service, installe le package `qiskit-ibm-transpiler` :

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

Le package s'authentifie automatiquement en utilisant tes [identifiants IBM Quantum Platform](/guides/cloud-setup), conformément à la gestion effectuée par [Qiskit Runtime](/guides/initialize-account) :
- Variable d'environnement : `QISKIT_IBM_TOKEN`
- Fichier de configuration `~/.qiskit/qiskit-ibm.json` (sous la section `default-ibm-quantum`).

*Remarque* : Ce package nécessite Qiskit SDK v1.X.

## Options de transpilation de qiskit-ibm-transpiler
- `backend_name` (optionnel, str) - Un nom de backend tel qu'attendu par QiskitRuntimeService (par exemple, `ibm_torino`). Si ce paramètre est défini, la méthode de transpilation utilise la disposition du backend spécifié pour l'opération de transpilation. Si une autre option ayant un impact sur ces paramètres est définie, comme `coupling_map`, les paramètres `backend_name` sont remplacés.
- `coupling_map` (optionnel, List[List[int]]) - Une liste de coupling map valide (par exemple, [[0,1],[1,2]]). Si ce paramètre est défini, la méthode de transpilation utilise cette coupling map pour l'opération de transpilation. S'il est défini, il remplace toute valeur spécifiée pour `target`.
- `optimization_level` (int) - Le niveau d'optimisation potentiel à appliquer lors du processus de transpilation. Les valeurs valides sont [1,2,3], où 1 correspond à l'optimisation la plus faible (et la plus rapide), et 3 à l'optimisation la plus poussée (et la plus longue).
- `ai` ("true", "false", "auto") - Indique s'il faut utiliser les fonctionnalités alimentées par l'IA lors de la transpilation. Les fonctionnalités IA disponibles peuvent inclure des passes de transpilation `AIRouting` ou d'autres méthodes de synthèse alimentées par l'IA. Si cette valeur est `"true"`, le service applique différentes passes de transpilation alimentées par l'IA en fonction du `optimization_level` demandé. Si `"false"`, il utilise les dernières fonctionnalités de transpilation Qiskit sans IA. Enfin, si `"auto"`, le service décide d'appliquer les passes heuristiques Qiskit standard ou les passes alimentées par l'IA en fonction de ton circuit.
- `qiskit_transpile_options` (dict) - Un objet dictionnaire Python pouvant inclure toute autre option valide dans la [méthode Qiskit `transpile()`](defaults-and-configuration-options). Si `qiskit_transpile_options` inclut `optimization_level`, il est ignoré au profit du `optimization_level` spécifié en paramètre d'entrée. Si `qiskit_transpile_options` inclut une option non reconnue par la méthode Qiskit `transpile()`, la bibliothèque génère une erreur.

Pour plus d'informations sur les méthodes `qiskit-ibm-transpiler` disponibles, consulte la [référence API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Pour en savoir plus sur l'API du service, consulte la [documentation REST API du Qiskit Transpiler Service.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## Exemples
Les exemples suivants montrent comment transpiler des circuits à l'aide du Qiskit Transpiler Service avec différents paramètres.

1. Crée un circuit et appelle le Qiskit Transpiler Service pour transpiler le circuit avec `ibm_torino` comme `backend_name`, 3 comme `optimization_level`, et sans utiliser l'IA lors de la transpilation.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Remarque* : tu ne peux utiliser que les appareils `backend_name` auxquels tu as accès avec ton compte IBM Quantum. En plus du `backend_name`, le `TranspilerService` accepte également `coupling_map` comme paramètre.

2. Produis un circuit similaire et transpile-le, en demandant les fonctionnalités de transpilation IA en définissant le drapeau `ai` à `True` :

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Produis un circuit similaire et transpile-le en laissant le service décider s'il faut utiliser les passes de transpilation alimentées par l'IA.